In [1]:
# Dependencies are installed in the demo-level Python 3.12 environment.

# Setup
Run `az login` before starting. A local `.env` file may define:

* `FOUNDRY_PROJECT_ENDPOINT` (optional; East US 2 training default)
* `FOUNDRY_EVALUATION_PROJECT_ENDPOINT` (optional; NCUS evaluation default)
* `X_FUNCTIONS_KEY`
* `AZURE_FUNCTIONS_ENDPOINT`

Foundry authentication uses Entra ID through `DefaultAzureCredential`.


In [2]:
from dotenv import load_dotenv

load_dotenv(override=True)

False

Create the Microsoft Foundry project client with Entra ID. The documented
`get_openai_client()` child supplies file, evaluation, and fine-tuning operations;
no OpenAI client is imported or constructed directly.


In [3]:
import json
import os
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = os.getenv(
    "FOUNDRY_PROJECT_ENDPOINT",
    "https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026",
)
EVALUATION_PROJECT_ENDPOINT = os.getenv(
    "FOUNDRY_EVALUATION_PROJECT_ENDPOINT",
    "https://ncus-prakharg-demo-2026.services.ai.azure.com/api/projects/ncus-prakharg-demo-2026",
)
NOTEBOOK_DIR = Path.cwd()
credential = DefaultAzureCredential()

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
foundry_client = project_client.get_openai_client()

evaluation_project_client = AIProjectClient(
    endpoint=EVALUATION_PROJECT_ENDPOINT,
    credential=credential,
)
evaluation_client = evaluation_project_client.get_openai_client()
print(f"> Created training project client for {PROJECT_ENDPOINT}")
print(f"> Created evaluation project client for {EVALUATION_PROJECT_ENDPOINT}")


> Created training project client for https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026
> Created evaluation project client for https://ncus-prakharg-demo-2026.services.ai.azure.com/api/projects/ncus-prakharg-demo-2026


In [4]:
# We'll create a "unique enough" identifier that lets us run this notebook
# multiple times and easily keep track of things each run creates.
import uuid

UNIQUE_ENOUGH_KEY = str(uuid.uuid4()).split("-")[0]
print(f"> Using unique enough key: {UNIQUE_ENOUGH_KEY}")

> Using unique enough key: aa31728f


In [5]:
DEVELOPER_PROMPT = """
You are an expert in arithmetic problem solving. Given a target number and a list of
numbers, your task is to combine all of the numbers exactly once using addition (+),
subtraction (-), multiplication (x), or division (/) to reach the target.

- You must use every number exactly once.
- Use parentheses as needed to control the order of operations.
- Return only a valid JSON object with the following exact syntax:
  {
    "expression": <the expression>,
    "result": <the result>
  }
- The expression should be a string representation of the mathematical expression.
- The result should be a string containing the integer result.
- If the exact target is not possible, return the closest valid result using all numbers.

# Example 1
target: 850
numbers: [100, 75, 50, 25, 6, 3]

Output:
{
  "expression": "((100 * 6) + (75 + 25) + (3 * 50))",
  "result": 850
}

# Example 2
target: 945
numbers: [25, 100, 9, 3, 6, 2]

Output:
{
  "expression": "((100 * 9) + (6 * 3) + (25 + 2))",
  "result": 945
}

# Example 3
target: 310
numbers: [7, 50, 75, 3, 8, 2]

Output:
{
  "expression": "((75 * 3) + (50 * 2) - (8 + 7))",
  "result": 310
}
""".strip()

# Baseline Evaluation
We're first going to assess how some models perform natively at this task given the above prompt.

In [6]:
# Load the held-out evaluation items. Inline file content avoids a service-side
# file-name loss in file-ID based evaluation runs.
import json

with open(NOTEBOOK_DIR / "data" / "countdown_eval_25.jsonl", encoding="utf-8") as stream:
    EVAL_ITEMS = [json.loads(line) for line in stream if line.strip()]

print(f"> Loaded {len(EVAL_ITEMS)} evaluation items.")


> Loaded 25 evaluation items.


In [7]:
# We need to describe the source and schema for our files.
EVAL_DATA_SOURCE = {
    "item_schema": {
        "type": "object",
        "properties": {
            "target": {"type": "integer"},
            "nums": {"type": "array", "items": {"type": "integer"}},
        },
    },
    "include_sample_schema": True,
    "type": "custom",
}

In [8]:
# Our expected Response schema from the models. We'll use structured outputs
# so the model must output only JSON.
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "math_expression",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"},
                "result": {"type": "string"},
            },
            "required": ["expression", "result"],
            "additionalProperties": False,
        },
    },
}

## Endpoint Grader
Our endpoint grader points to a remote endpoint that hosts the grader function.

It has an API surface that's basically the Python grader, but via POST'ing JSON
to an endpoint.

We're using an Azure Function with key based auth, so each request will need
the `X-Functions-Key` header with our authentication key.

In [9]:
import os

URL = "https://prg-rft-countdown-ncus.azurewebsites.net/api/grader"
function_key = os.environ.get("X_FUNCTIONS_KEY")
if not function_key:
    raise RuntimeError("X_FUNCTIONS_KEY must be supplied through the process environment.")

ENDPOINT_GRADER = {
    "type": "endpoint",
    "name": "Remote Countdown Grader",
    "url": URL,
    "headers": {"X-Functions-Key": function_key},
    "pass_threshold": 1.0,
}
print(f"> Defined endpoint grader for {URL}")


> Defined endpoint grader for https://prg-rft-countdown-ncus.azurewebsites.net/api/grader


Let's now make sure our Endpoint Grader is reachable. If it's not, there's no sense in
submitting an Eval or RFT job. We can POST some sample input and check if we get back
a score.

In [10]:
# Manually test the grader.
import requests

data = {
    "item": {"target": 47, "nums": [86, 22, 88, 72]},
    "sample": {
        "output_json": {
            "expression": "(72 + 22) / (88 - 86)",
            "result": 47,
        },
    },
}

result = requests.post(
    URL,
    headers={
        "Content-Type": "application/json",
        "X-Functions-Key": os.getenv("X_FUNCTIONS_KEY"),
    },
    json=data,
)

print(f"> Got HTTP {result.status_code}")
if result.status_code == 200:
    score = result.json()["score"]
    print(f"> Score is {score}.")
    if score != 1.0:
        raise RuntimeError("Uh oh...you might want to check your grader!")

> Got HTTP 200
> Score is 1.0.


For comparison, we'll use the same code but run as a Python grader.

In practice, this makes no sense...but it demonstrates how similar the two
graders are in shape.

The code is in [grader.py](./grader.py), so we can import the module and
use Python's native `inspect` module to dump it as source code into a
string we pass to the grader.

In [11]:
# Load the exact source submitted to the Foundry Python grader validator.
from pprint import pprint

code = (NOTEBOOK_DIR / "grader.py").read_text(encoding="utf-8")

PYTHON_GRADER = {
    "type": "python",
    "name": "Python Countdown Grader",
    "source": code,
    "pass_threshold": 1.0,
}
print("> Defined Python grader")
pprint(PYTHON_GRADER)

> Defined Python grader
{'name': 'Python Countdown Grader',
 'pass_threshold': 1.0,
 'source': '"""\n'
           'Python Grader logic for OpenAI Evals\n'
           '\n'
           'Example from '
           'https://github.com/azure-ai-foundry/fine-tuning/blob/main/Demos/RFT_Countdown/demo_with_python_grader.ipynb\n'
           '"""\n'
           'from typing import Any, Dict\n'
           'import ast\n'
           'import json\n'
           'import logging\n'
           'import re\n'
           '\n'
           '\n'
           'def _eval(n):\n'
           '    if isinstance(n, ast.Constant):\n'
           '        return n.value\n'
           '\n'
           '    if isinstance(n, ast.BinOp) and type(n.op) in {\n'
           '        ast.Add: lambda a, b: a + b,\n'
           '        ast.Sub: lambda a, b: a - b,\n'
           '        ast.Mult: lambda a, b: a * b,\n'
           '        ast.Div: lambda a, b: a / b,\n'
           '        ast.FloorDiv: lambda a, b: a // b,\n'
        

Now we're ready to define our Eval.

In [ ]:
# Create our Eval
baseline_eval = evaluation_client.evals.create(
    name=f"endpoint-countdown-eval-{UNIQUE_ENOUGH_KEY}",
    data_source_config=EVAL_DATA_SOURCE,
    testing_criteria=[PYTHON_GRADER, ENDPOINT_GRADER],
)
print(f"> Created eval {baseline_eval.id}: {baseline_eval.name}")

In [13]:
# Define our Runs and monitor each evaluation to a terminal state.
import time

USER_PROMPT = """
target: {{item.target}}
numbers: {{item.nums}}
""".strip()

RUNS = []
for model in ["gpt-4.1", "o4-mini"]:
    RUN_DATA_SOURCE = {
        "type": "responses",
        "model": model,
        "source": {"type": "file_content", "content": EVAL_ITEMS},
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "type": "message",
                    "role": "developer",
                    "content": DEVELOPER_PROMPT,
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": USER_PROMPT,
                },
            ],
        },
        "sampling_params": {"text": {"format": RESPONSE_SCHEMA}},
    }
    run = evaluation_client.evals.runs.create(
        name=f"endpoint-countdown-eval-run-{model}-{UNIQUE_ENOUGH_KEY}",
        eval_id=baseline_eval.id,
        data_source=RUN_DATA_SOURCE,
    )
    print(f"> Created run {run.id}: {run.name}")
    RUNS.append(run)

terminal_eval_statuses = {"completed", "failed", "canceled", "cancelled"}
remaining_eval_runs = {run.id for run in RUNS}
FINAL_EVAL_RUNS = {}
while remaining_eval_runs:
    for run_id in list(remaining_eval_runs):
        current = evaluation_client.evals.runs.retrieve(run_id, eval_id=baseline_eval.id)
        print(f"> Eval run {run_id}: {current.status}")
        if current.status in terminal_eval_statuses:
            FINAL_EVAL_RUNS[run_id] = current
            remaining_eval_runs.remove(run_id)
    if remaining_eval_runs:
        time.sleep(30)

print(f"> {len(FINAL_EVAL_RUNS)} evaluation runs reached terminal states.")


> Created run evalrun_5ecd9cbf2dac4c10879ee2e96310c9bc: endpoint-countdown-eval-run-gpt-4.1-aa31728f


> Created run evalrun_891fa28cb8b141b69da9990cde2ed1c7: endpoint-countdown-eval-run-o4-mini-aa31728f


> Eval run evalrun_5ecd9cbf2dac4c10879ee2e96310c9bc: in_progress


> Eval run evalrun_891fa28cb8b141b69da9990cde2ed1c7: in_progress


> Eval run evalrun_5ecd9cbf2dac4c10879ee2e96310c9bc: failed


> Eval run evalrun_891fa28cb8b141b69da9990cde2ed1c7: failed
> 2 evaluation runs reached terminal states.


# Reinforcement Fine Tuning
Now let's train a model to improve upon our out-of-box o4-mini experience.

In [14]:
# Upload fresh copies so this run cannot reuse files from earlier failed attempts.
def upload_file(path: Path, purpose: str):
    with path.open("rb") as stream:
        uploaded = foundry_client.files.create(file=stream, purpose=purpose)
    uploaded = foundry_client.files.wait_for_processing(uploaded.id)
    print(f"> Freshly uploaded {path.name}: {uploaded.id}")
    return uploaded

training_file = upload_file(
    NOTEBOOK_DIR / "data" / "countdown_train_25.jsonl", "fine-tune"
)
validation_file = upload_file(
    NOTEBOOK_DIR / "data" / "countdown_valid_10.jsonl", "fine-tune"
)


> Reusing countdown_train_100.jsonl: file-170698a3c4e54cea8985b6b1b0a16a44


> Reusing countdown_valid_50.jsonl: file-d2fb9bd928b7452f88350579469e4346


We can drop the `pass_threshold` settings from the individual graders. We can set it on
the `multi_grader` we'll use to combine both Python and Endpoint graders.

In [15]:
import copy
import time

RFT_ENDPOINT_GRADER = copy.deepcopy(ENDPOINT_GRADER)
RFT_ENDPOINT_GRADER.pop("pass_threshold", None)
RFT_PYTHON_GRADER = copy.deepcopy(PYTHON_GRADER)
RFT_PYTHON_GRADER.pop("pass_threshold", None)

if len(RFT_PYTHON_GRADER["source"].strip()) < 100:
    raise RuntimeError("Python grader source is empty or unexpectedly short.")

schema = RESPONSE_SCHEMA["json_schema"]
if schema["name"] != "math_expression":
    raise RuntimeError("Response schema name must be math_expression.")
if schema["schema"]["properties"] != {
    "expression": {"type": "string"},
    "result": {"type": "string"},
}:
    raise RuntimeError("Response schema properties are not the concrete countdown schema.")

RFT_MULTI_GRADER = {
    "type": "multi",
    "name": "Combined local and remote grader",
    "graders": {
        "python": RFT_PYTHON_GRADER,
        "endpoint": RFT_ENDPOINT_GRADER,
    },
    "calculate_output": "(python + endpoint) / 2",
}

job_payload = {
    "suffix": f"ncus-countdown-payload-{UNIQUE_ENOUGH_KEY}",
    "model": "o4-mini-2025-04-16",
    "training_file": training_file.id,
    "validation_file": validation_file.id,
    "method": {
        "type": "reinforcement",
        "reinforcement": {
            "grader": RFT_MULTI_GRADER,
            "response_format": RESPONSE_SCHEMA,
        },
    },
}

job = foundry_client.fine_tuning.jobs.create(**job_payload)
print(f"> Created RFT job {job.id}: {job.status}")

while job.status not in {"succeeded", "failed", "cancelled"}:
    time.sleep(60)
    job = foundry_client.fine_tuning.jobs.retrieve(job.id)
    print(f"> RFT job {job.id}: {job.status}")

print(f"> Terminal RFT job {job.id}: {job.status}")
print(f"> Fine-tuned model: {job.fine_tuned_model}")
print(f"> Error: {job.error}")


> Created RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857
{
  "id": "ftjob-56888d9e8f9d417e83c99d0e71c5e857",
  "created_at": 1791059932,
  "model": "o4-mini-2025-04-16",
  "object": "fine_tuning.job",
  "seed": 841177956,
  "status": "pending",
  "training_file": "file-170698a3c4e54cea8985b6b1b0a16a44",
  "validation_file": "file-d2fb9bd928b7452f88350579469e4346",
  "estimated_finish": 1791061612,
  "metadata": {
    "base_model": "o4-mini",
    "model_version": "2025-04-16",
    "created_by": "Prakhar Gupta",
    "created_by_email": "prakharg@microsoft.com"
  },
  "method": {
    "type": "reinforcement",
    "reinforcement": {
      "grader": {
        "name": "Combined local and remote grader",
        "type": "multi",
        "graders": {
          "python": {
            "type": "python",
            "image_tag": "2025-05-08",
            "source": "\"\"\"\nPython Grader logic for OpenAI Evals\n\nExample from https://github.com/azure-ai-foundry/fine-tuning/blob/main/Demos/RFT_Count

C:\Data\AIP\fine-tuning\Demos\Agentic_RFT_PrivatePreview\.venv\Lib\site-packages\pydantic\main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `literal['string_check']` - serialized value may not be as expected [field_name='type', input_value='multi', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `TextSimilarityGrader` - serialized value may not be as expected [field_name='grader', input_value=StringCheckGrader(input=N...python + endpoint) / 2'), input_type=StringCheckGrader])
  PydanticSerializationUnexpectedValue(Expected `PythonGrader` - serialized value may not be as expected [field_name='grader', input_value=StringCheckGrader(input=N...python + endpoint) / 2'), input_type=StringCheckGrader])
  PydanticSerializationUnexpectedValue(Expected `ScoreModelGrader` - serialized value may not be as expected [field_name='grader', input_value=StringCheckGrader(input=N...python + endpoint) / 2'), input_type=StringCheckGrade

> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


...

> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: pending


> RFT job ftjob-56888d9e8f9d417e83c99d0e71c5e857: cancelled
> Terminal RFT job:
{
  "id": "ftjob-56888d9e8f9d417e83c99d0e71c5e857",
  "created_at": 1791059932,
  "finished_at": 1791067699,
  "model": "o4-mini-2025-04-16",
  "object": "fine_tuning.job",
  "seed": 841177956,
  "status": "cancelled",
  "training_file": "file-170698a3c4e54cea8985b6b1b0a16a44",
  "validation_file": "file-d2fb9bd928b7452f88350579469e4346",
  "estimated_finish": 1791061612,
  "metadata": {
    "base_model": "o4-mini",
    "model_version": "2025-04-16",
    "created_by": "Prakhar Gupta",
    "created_by_email": "prakharg@microsoft.com"
  },
  "method": {
    "type": "reinforcement",
    "reinforcement": {
      "grader": {
        "name": "Combined local and remote grader",
        "type": "multi",
        "graders": {
          "python": {
            "type": "python",
            "image_tag": "2025-05-08",
            "source": "\"\"\"\nPython Grader logic for OpenAI Evals\n\nExample from https://github.com/